# NDC — Colab Pro+ / single A100

Select A100 in Runtime → Change runtime type, upload this notebook, then Run all.
Authorize Google Drive when prompted. Dataset: ndhoang2310/phenobench-dataset, version1.
If Kaggle asks for authentication, use your own Kaggle token (never put it in the notebook).

Default runs P2 Control and NDC-Ex1 sequentially for a matched execution comparison.
Model, FP32, crops, WDice, Adam, LR/scheduler and100epochs/val5 remain unchanged.
Single A100: two paired batch4 forwards per optimizer step (effective8), rank0 BN
buffer semantics and independent Dropout streams;176steps/epoch,768validation images.
No AMP/TF32/checkpointing/fallback. This is not bitwise replay of historical DDP.
Each completed epoch is backed up to Drive, with checksums and exact-source resume.
Final output: one ZIP per mode. A100 acceptance is verified before real training.


In [ ]:
# FIRST CONFIG CELL
REPO_URL = 'https://github.com/FUAVX/RPD_Reproduce.git'
GIT_REF = 'f/fuse-ndc'
SOURCE_COMMIT = ''  # fresh run: branch tip; resume: use exact SHA from backup
EXPERIMENT_MODES = ['control', 'ex1']  # paired Control is recommended for fairness
RUN_TAG = 'colab_a100_run1'
DATASET_HANDLE = 'ndhoang2310/phenobench-dataset/versions/1'
DRIVE_ROOT = '/content/drive/MyDrive/RPD_Experiments/NDC_Colab_A100'
WORKING_ROOT = '/content/ndc_colab'
NUM_WORKERS = 2
RESUME = True  # restore verified last completed epoch, if available
EXPERIMENTS = {'control':'NDC-Control','ex1':'NDC-Ex1'}
if not EXPERIMENT_MODES or len(set(EXPERIMENT_MODES))!=len(EXPERIMENT_MODES) or any(m not in EXPERIMENTS for m in EXPERIMENT_MODES):
    raise ValueError('Select control and/or ex1; no other NDC variants')


In [ ]:
# Retain Colab torch/torchvision/CUDA. Install the same legacy training stack.
!nvidia-smi
!pip install -q 'numpy<2' 'pytorch-lightning==1.9.5' 'torchmetrics==0.11.4' 'setuptools<70' tensorboard matplotlib pandas tabulate timm scikit-image tifffile albumentations pyyaml rich kagglehub


In [ ]:
import os,json,pathlib,subprocess,shutil,hashlib,copy,sys
# Applies to every child process, including export/profile.
os.environ['NVIDIA_TF32_OVERRIDE'] = '0'
from google.colab import drive
drive.mount('/content/drive')
ROOT=pathlib.Path(WORKING_ROOT);ROOT.mkdir(parents=True,exist_ok=True)
PERSISTENT=pathlib.Path(DRIVE_ROOT);PERSISTENT.mkdir(parents=True,exist_ok=True)
REPO_DIR=ROOT/'RPD_P2'
BASE_COMMIT='d77d518b8f650443216070fa76742db76e41b8ac'
PROTOCOL='ndc_colab_paired_batch4_v1'
backup_metas={}
for mode in EXPERIMENT_MODES:
    pointer=PERSISTENT/f'{EXPERIMENTS[mode]}_{RUN_TAG}'/'latest.json'
    if RESUME and pointer.exists():
        meta=json.loads(pointer.read_text())
        if meta['protocol']!=PROTOCOL:raise ValueError('Backup protocol mismatch')
        backup_metas[mode]=meta
pins={m['source_commit'] for m in backup_metas.values()}
if len(pins)>1:raise ValueError('Selected backups have different source SHAs; resume them separately')
resume_sha=next(iter(pins),None)
if SOURCE_COMMIT and resume_sha and not resume_sha.startswith(SOURCE_COMMIT):raise ValueError('SOURCE_COMMIT conflicts with backup')
requested_sha=resume_sha or SOURCE_COMMIT

def run(command,cwd=REPO_DIR,log=None):
    if command[0]==sys.executable and command[1].endswith('.py'):
        command=[sys.executable,'scripts/ndc_colab_exec.py',*command[1:]]
    print('+',' '.join(map(str,command)),flush=True)
    stream=open(log,'w') if log else None
    try:
        with subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
            for line in process.stdout:
                print(line,end='',flush=True)
                if stream:stream.write(line);stream.flush()
            code=process.wait()
        if code:raise subprocess.CalledProcessError(code,command)
    finally:
        if stream:stream.close()
if REPO_DIR.exists():
    actual=subprocess.check_output(['git','remote','get-url','origin'],cwd=REPO_DIR,text=True).strip()
    if actual!=REPO_URL:raise RuntimeError('Existing clone has a different origin; use a fresh runtime')
    if subprocess.check_output(['git','status','--porcelain'],cwd=REPO_DIR,text=True).strip():raise RuntimeError('Existing clone is modified')
    run(['git','fetch','origin',GIT_REF])
    run(['git','checkout','--detach',requested_sha or 'FETCH_HEAD'])
else:
    run(['git','clone','--branch',GIT_REF,'--single-branch',REPO_URL,str(REPO_DIR)],cwd=ROOT)
    if requested_sha:run(['git','checkout','--detach',requested_sha])
run(['git','merge-base','--is-ancestor',BASE_COMMIT,'HEAD'])
source_sha=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO_DIR,text=True).strip()
for name in ['utils/ndc_colab.py','scripts/ndc_colab_exec.py','test_ndc_colab.py','scripts/preflight_ndc_colab.py','scripts/check_ndc_data.py','scripts/check_ndc_run.py','scripts/export_ndc.py']:
    if not (REPO_DIR/name).is_file():raise FileNotFoundError(name)
sys.path.insert(0,str(REPO_DIR))
import torch,yaml
from utils.ndc_colab import configure_fp32,restore_backup,publish_backup,validate_recipe
configure_fp32()
if not torch.cuda.is_available() or torch.cuda.device_count()!=1 or 'A100' not in torch.cuda.get_device_name(0):
    raise RuntimeError('Select one A100; this notebook does not substitute a smaller GPU')
GPU_NAME=torch.cuda.get_device_name(0)
GPU_VRAM=torch.cuda.get_device_properties(0).total_memory
print('Source:',source_sha,'GPU:',GPU_NAME,'VRAM GiB:',GPU_VRAM/2**30)


In [ ]:
# Download to local VM storage, not Google Drive. Version pinned to1.
import kagglehub
try:
    download_root=pathlib.Path(kagglehub.dataset_download(DATASET_HANDLE))
except Exception as error:
    status=getattr(getattr(error,'response',None),'status_code',None)
    if status not in (401,403):raise
    kagglehub.login()
    download_root=pathlib.Path(kagglehub.dataset_download(DATASET_HANDLE))
def is_root(p):return all((p/part).is_dir() for part in ['train/images','train/semantics','val/images','val/semantics'])
candidates=[download_root,download_root/'PhenoBench',download_root/'PhenoBench-v110'/'PhenoBench']
candidates.extend(p.parent for p in download_root.rglob('train') if p.is_dir())
dataset_dir=next((p for p in candidates if is_root(p)),None)
if dataset_dir is None:raise FileNotFoundError('PhenoBench folder structure not found in downloaded dataset')
run([sys.executable,'scripts/check_ndc_data.py','--dataset-dir',str(dataset_dir),'--output',str(ROOT/'dataset_audit.json')])
for script in ['test_wdice.py','test_validation_infrastructure.py','test_ndc.py','test_ndc_colab.py']:
    run([sys.executable,script],log=ROOT/f'{script}.log')


In [ ]:
# Prepare immutable per-mode configs. Existing Kaggle notebook/YAML are unchanged.
base=yaml.safe_load((REPO_DIR/'configs/p2_scenarios/P2_Ex7_RepDWNet_WDice-Loss.yaml').read_text())
RUNS={}
for mode in EXPERIMENT_MODES:
    scenario=EXPERIMENTS[mode];run_dir=ROOT/f'{scenario}_{RUN_TAG}';run_dir.mkdir(exist_ok=True)
    training_dir=run_dir/'training_output';backup_dir=PERSISTENT/f'{scenario}_{RUN_TAG}'
    if mode in backup_metas:
        if training_dir.exists():
            previous=ROOT/'previous_local_outputs'/f'{scenario}_{RUN_TAG}'
            previous.parent.mkdir(exist_ok=True)
            if previous.exists():raise FileExistsError('Use a fresh Colab runtime to avoid overwriting previous local evidence')
            training_dir.rename(previous)
        restore_backup(backup_dir,training_dir)
    elif training_dir.exists() and any(training_dir.iterdir()):
        raise FileExistsError('Local output exists without selected backup; choose a fresh runtime or RUN_TAG')
    cfg=yaml.safe_load((REPO_DIR/'configs/p3_fuse_ndc'/f'{scenario}.yaml').read_text())
    inherited=copy.deepcopy(cfg);inherited['backbone'].pop('nested',None);inherited['experiment']=base['experiment']
    assert inherited==base,'Recipe drift beyond NDC'
    cfg['data'].update(path_to_dataset=str(dataset_dir),num_workers=NUM_WORKERS)
    cfg['train'].update(devices=1,accelerator='gpu',accumulate_grad_batches=2)
    cfg['execution']={'protocol':PROTOCOL,'backup_dir':str(backup_dir),
                      'hardware':'single_A100','historical_world_size':2,'microbatch_size':4,'effective_batch':8,
                      'train_steps_per_epoch':176,'validation_images':768,'TF32':False}
    validate_recipe(cfg)
    if mode in backup_metas:
        previous=yaml.safe_load((training_dir/'resolved_config.yaml').read_text())
        if previous!=cfg:raise ValueError('Resume config differs from backed-up recipe; do not silently change data/workers/protocol')
    config_path=run_dir/'runtime_config.yaml';config_path.write_text(yaml.safe_dump(cfg,sort_keys=False))
    RUNS[mode]={'dir':run_dir,'training':training_dir,'backup':backup_dir,'config':config_path,'cfg':cfg}
print('Prepared:',list(RUNS))


In [ ]:
# Mandatory native A100 acceptance. No activation checkpointing or recipe fallback.
PREFLIGHT_PASSED=False
for mode,info in RUNS.items():
    run([sys.executable,'scripts/preflight_ndc_colab.py','--config',str(info['config']),
         '--output',str(info['dir']/'native_a100_preflight.json')],log=info['dir']/'native_a100_preflight.log')
    if json.loads((info['dir']/'native_a100_preflight.json').read_text())['status']!='passed':raise RuntimeError('Native preflight failed')
run([sys.executable,'scripts/profile_ndc.py','--device','cuda','--experiments',*EXPERIMENT_MODES,
     '--output',str(ROOT/'complexity_native_shapes.json')])
PREFLIGHT_PASSED=True


In [ ]:
if not globals().get('PREFLIGHT_PASSED',False):raise RuntimeError('Native preflight must pass')
for mode,info in RUNS.items():
    last=info['training']/'checkpoints/last.ckpt'
    command=[sys.executable,'multi_metric_train.py','--config',str(info['config']),
             '--export_dir',str(info['training']),'--devices','1']
    completed=False
    if last.exists():
        state=torch.load(last,map_location='cpu',weights_only=False)
        if state.get('execution_protocol')!=PROTOCOL or state.get('nested')!=info['cfg']['backbone'].get('nested'):
            raise ValueError('Wrong protocol or model in resume checkpoint')
        if (info['training']/'source_commit.txt').read_text().strip()!=source_sha:raise ValueError('Wrong source revision')
        completed=state['epoch']==99 and state['global_step']==17600
        if not completed:command+=['--ckpt_path',str(last),'--resume']
        del state
    if not completed:run(command,log=info['dir']/'colab_training_console.log')
    run([sys.executable,'scripts/check_ndc_run.py','--output-dir',str(info['training']),
         '--report',str(info['dir']/'campaign_audit.json')])
    state=torch.load(last,map_location='cpu',weights_only=False)
    if state['global_step']!=17600:raise ValueError('Expected17600 optimizer steps')
    del state
    publish_backup(info['training'],info['backup'])


In [ ]:
# Strict unfused exports + final ZIP per mode, copied to Drive.
import importlib.metadata
for mode,info in RUNS.items():
    for policy in ['best_mIoU','best_weedIoU']:
        matches=list((info['training']/'checkpoints').glob(f'*{policy}*.ckpt'))
        if len(matches)!=1:raise RuntimeError(f'Expected one {policy} checkpoint')
        target=info['dir']/'main_export'/policy
        if target.exists():
            report=json.loads((target/'export_report.json').read_text())
            if report['checkpoint_sha256']!=hashlib.sha256(matches[0].read_bytes()).hexdigest():raise ValueError('Existing export is from a different checkpoint')
        else:
            run([sys.executable,'scripts/export_ndc.py','--config',str(info['config']),'--checkpoint',str(matches[0]),
                 '--output-dir',str(target),'--device','cuda'])
    shutil.copy2(ROOT/'dataset_audit.json',info['dir']/'dataset_audit.json')
    shutil.copy2(ROOT/'complexity_native_shapes.json',info['dir']/'complexity_native_shapes.json')
    manifest={'status':'completed','experiment':EXPERIMENTS[mode],'run_tag':RUN_TAG,'source_repository':REPO_URL,
        'source_commit':source_sha,'base_commit':BASE_COMMIT,'dataset_handle':DATASET_HANDLE,'execution_protocol':PROTOCOL,
        'resolved_config':info['cfg'],'hardware':{'gpu':GPU_NAME,'VRAM_bytes':GPU_VRAM,'world_size':1},
        'precision':'FP32, TF32 disabled','microbatch':4,'paired_microbatches_per_step':2,'effective_batch':8,
        'optimizer_steps_per_epoch':176,'optimizer_steps_total':17600,'validation_images':768,
        'BN':'batch4; restore pre-pair buffers for logical rank1; retain rank0 post-forward buffers',
        'Dropout':'two independent persisted RNG streams; not bitwise replay of historical dataloader/CUDA',
        'fairness_limitations':'same objective/BN batch/steps/validation set; hardware, worker RNG and reduction arithmetic differ; matched Control recommended; do not compare raw training time across hardware',
        'matched_control_requested':'control' in EXPERIMENT_MODES,
        'native_preflight':json.loads((info['dir']/'native_a100_preflight.json').read_text()),
        'campaign_audit':json.loads((info['dir']/'campaign_audit.json').read_text()),
        'versions':{name:importlib.metadata.version(name) for name in ['torch','torchvision','pytorch-lightning','torchmetrics','numpy','kagglehub']}}
    (info['dir']/'run_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
    archive=pathlib.Path(shutil.make_archive(str(ROOT/f'{EXPERIMENTS[mode]}_{RUN_TAG}_artifacts'),'zip',root_dir=info['dir']))
    final_dir=PERSISTENT/'completed';final_dir.mkdir(exist_ok=True)
    shutil.copy2(archive,final_dir/archive.name)
    if hashlib.sha256(archive.read_bytes()).hexdigest()!=hashlib.sha256((final_dir/archive.name).read_bytes()).hexdigest():raise IOError('Final Drive ZIP copy failed checksum')
    print('[DONE]',final_dir/archive.name)
